# Inference
> Load finetuned model and infer on single image

In [ ]:
#| default_exp inference.decoder_inference

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2


In [ ]:
!which python

In [ ]:
#| export
import sys
from pathlib import Path
import os
from fastcore.all import *
import shutil


In [ ]:
#| export
CURRENT_NB='/home/hpc_storage/data/projects/userA/sam3_fine_tune/nbs'

In [ ]:
Path.cwd()

In [ ]:
##os.chdir('/home/hpc_storage/data/projects/userA/sam3/sam3/')
#import os
#os.chdir(CURRENT_NB)

In [ ]:

#X:\data\projects\userB\data\universal_model\ThinWire_Segmentation\train_images_18122025_gatepads_split\train
path_dir = Path(r'/home/hpc_storage/data/projects/userB/data/universal_model/ThinWire_Segmentation/train_images_18122025_gatepads_split/train')

print(f'path exists: {path_dir.exists()}')

In [ ]:
experiment_name = '8cat_bentwire_separate'
experiment_name = '9cat_diverse_wires_v1_2gpu'
#ckpnt = Path('/home/hpc_storage/data/projects/userB/test_sam3/sam3_fine_tune', f'experiments/{experiment_name}/checkpoints/val_pin_detection_coco_eval_segm_AP.pt')
ckpnt = Path('/home/hpc_storage/data/projects/userB/test_sam3/sam3_fine_tune', f'experiments/{experiment_name}/checkpoints/checkpoint_10.pt')

In [ ]:
print(f'checkpoint exists: {ckpnt.exists()}')

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
import sam3
from PIL import Image
from sam3 import build_sam3_image_model
from sam3.model.box_ops import box_xywh_to_cxcywh
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.visualization_utils import draw_box_on_image, normalize_bbox, plot_results


In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# use bfloat16 for the entire notebook
torch.autocast("cuda", dtype=torch.bfloat16).__enter__()

In [ ]:
def load_model(bpe_path: str, sam3_checkpoint: Path, finetuned_checkpoint: Path, 
               device: torch.device = None) -> torch.nn.Module:
    """Load SAM3 model with fine-tuned weights.
     Args:
         bpe_path: Path to the BPE tokenizer file.
         sam3_checkpoint: Path to the original SAM3 model checkpoint.
         finetuned_checkpoint: Path to the fine-tuned model checkpoint.
         device: Optional torch device to load the model on. If None, it will automatically use CUDA if available.
     Returns:
         Loaded SAM3 model with fine-tuned weights."""
    
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    model = build_sam3_image_model(bpe_path=bpe_path, checkpoint_path=sam3_checkpoint)
    finetuned_weights = torch.load(finetuned_checkpoint, map_location=device,weights_only=False)
    model.load_state_dict(finetuned_weights['model'])
    model.eval()
    
    print("Model loaded")
    return model

In [ ]:
#| export
def run_inference(model, image_path, prompt):
    if isinstance(image_path, (str, Path)):
        image = Image.open(image_path)
    else:
        image = image_path
    width, height = image.size
    processor = Sam3Processor(model, confidence_threshold=0.5)
    inference_state = processor.set_image(image)
    processor.reset_all_prompts(inference_state)
    inference_state = processor.set_text_prompt(state=inference_state, prompt=prompt)
    img0 = image
    plot_results(img0, inference_state)

In [ ]:
bpe_path = '/home/hpc_storage/data/projects/userA/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz'
sam3_checkpoint = Path('/home/hpc_storage/data/projects/userA/sam3/models--facebook--sam3/sam3.pt')
finetuned_checkpoint = Path("/home/hpc_storage/Models/sam3-finetune/2026-07-29/sam3_finetuned_9cat_diverse_wires_v1.pt")
model = load_model(bpe_path,sam3_checkpoint, ckpnt, device=device)


In [ ]:
image_path = Path(data_path)
imgs = [i for i in image_path.ls() if ('nonroi' not in Path(i).name and 'mask' not in Path(i).name)]
print(f'Found {len(imgs)} images in {data_path}')

In [ ]:
import numpy as np
idx = np.random.randint(0, len(imgs))

run_inference(model, imgs[idx], prompt="thick_wire_panda")

In [ ]:

run_inference(model, imgs[idx], prompt="thick_wire_panda")

In [ ]:
run_inference(model, imgs[idx], prompt="thick_wire_panda")

In [ ]:
new_img_path= Path(r'/home/hpc_storage/data/projects/userB/data/cegled_econo/47834/curated8bit/test/images')
new_img_path.exists()

In [ ]:
import numpy as np
idx = np.random.randint(0, len(new_img_path.ls()) - 1)
sm_im =Path(new_img_path.ls()[idx])
run_inference(model, sm_im, prompt="matrix_chip")

In [ ]:
new_path = Path(r'/home/hpc_storage/data/projects/userB/data/cegled_econo/43967/new_16bit_processed')
new_path.exists()

In [ ]:
idx = np.random.randint(0, len(new_path.ls()) - 1)
sm_nw_p = Path(new_path.ls()[idx])
run_inference(model, sm_nw_p, prompt="matrix_chip")

### SAKI chips

In [ ]:
from glob import glob


In [ ]:
d_p = Path(r'/home/hpc_storage/data/projects/userB/data/SAKI_processed/')
d_p.exists()




In [ ]:

image_exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp'}
imgs = [p for p in d_p.rglob('*') if p.suffix.lower() in image_exts]

In [ ]:
len(imgs)

In [ ]:
from tqdm.auto import tqdm

In [ ]:
idx = np.random.randint(0, len(imgs))
sm_nw_p = imgs[idx]
run_inference(model, sm_nw_p, prompt="matrix_chip")

In [ ]:
import numpy as np
from PIL import Image, ImageDraw

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

def visualize_and_save(image_path, inference_state, save_path=None, binary=False):
    """
    Visualize image, masks, and overlay with consistent colors per category.
    
    Args:
        image_path: Path to the original image
        inference_state: Inference state containing predictions
        save_path: Optional path to save the figure
        binary: If True, use black/white masks only
    """
    image = np.array(Image.open(image_path).convert("RGB"))
    h, w = image.shape[:2]
    
    # Extract boxes, labels, scores from inference_state
    boxes = inference_state.get('pred_boxes', [])
    labels = inference_state.get('pred_labels', [])
    scores = inference_state.get('pred_scores', [])
    masks = inference_state.get('pred_masks', None)
    
    # Build color map per unique category
    unique_labels = list(set(labels)) if labels else []
    if binary:
        color_map = {lbl: (1.0, 1.0, 1.0) for lbl in unique_labels}
    else:
        palette = list(mcolors.TABLEAU_COLORS.values())
        color_map = {lbl: mcolors.to_rgb(palette[i % len(palette)]) 
                     for i, lbl in enumerate(unique_labels)}

    # Create composite mask overlay
    overlay = image.copy().astype(np.float32)
    mask_canvas = np.zeros((h, w, 3), dtype=np.float32)

    if masks is not None:
        for i, (mask, label) in enumerate(zip(masks, labels)):
            color = color_map.get(label, (1.0, 1.0, 1.0))
            for c in range(3):
                mask_canvas[:, :, c] += mask * color[c]
            # Blend into overlay
            alpha = 0.4
            for c in range(3):
                overlay[:, :, c] = np.where(
                    mask > 0.5,
                    overlay[:, :, c] * (1 - alpha) + color[c] * 255 * alpha,
                    overlay[:, :, c]
                )

    mask_canvas = np.clip(mask_canvas, 0, 1)
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(image)
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(mask_canvas if not binary else mask_canvas.mean(axis=2), 
                   cmap='gray' if binary else None)
    axes[1].set_title("Masks")
    axes[1].axis("off")

    axes[2].imshow(overlay)
    # Draw boxes
    for box, label, score in zip(boxes, labels, scores):
        color = color_map.get(label, (1.0, 1.0, 1.0))
        x1, y1, x2, y2 = box
        rect = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                   linewidth=2, edgecolor=color, facecolor='none')
        axes[2].add_patch(rect)
        axes[2].text(x1, y1 - 5, f"{label} {score:.2f}", color=color,
                     fontsize=8, backgroundcolor='black')
    axes[2].set_title("Overlay")
    axes[2].axis("off")

    # Legend
    legend_patches = [mpatches.Patch(color=color_map[lbl], label=lbl) 
                      for lbl in unique_labels]
    if legend_patches:
        axes[2].legend(handles=legend_patches, loc='lower right', fontsize=8)

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
        print(f"Saved to {save_path}")

    plt.show()
    return fig


In [ ]:

# Example usage:
visualize_and_save(sm_im, inference_state, save_path=None, binary=False)

In [ ]:
run_inference(model, sm_im, prompt="matrix_chip")

### Patch-wise inference for large images
Run SAM3 on overlapping patches, stitch the predicted masks back to the original image size, and optionally overlay or save the result.

In [ ]:
#| export
def _extract_patches(image: Image.Image, patch_size: int, overlap: int):
    """Extract overlapping patches from a PIL image. Returns list of (x, y, patch)."""
    w, h = image.size
    if w <= patch_size and h <= patch_size:
        return [(0, 0, image)]
    stride = patch_size - overlap
    ys = list(range(0, h - patch_size + 1, stride))
    if not ys or ys[-1] + patch_size < h:
        ys.append(h - patch_size)
    xs = list(range(0, w - patch_size + 1, stride))
    if not xs or xs[-1] + patch_size < w:
        xs.append(w - patch_size)
    patches = []
    for y in ys:
        for x in xs:
            patches.append((x, y, image.crop((x, y, x + patch_size, y + patch_size))))
    return patches


def _distance_weight(ph: int, pw: int):
    """Return a weight map high at the patch center and tapering to the borders."""
    yy = np.arange(ph, dtype=np.float32)
    xx = np.arange(pw, dtype=np.float32)
    dist_y = np.minimum(yy + 0.5, ph - yy - 0.5)
    dist_x = np.minimum(xx + 0.5, pw - xx - 0.5)
    Y, X = np.meshgrid(dist_y, dist_x, indexing="ij")
    return np.minimum(X, Y)


def _stitch_patch_masks(patch_masks, image_h: int, image_w: int):
    """Stitch patch masks back into a full-size soft mask using distance-weighted blending."""
    acc = np.zeros((image_h, image_w), dtype=np.float64)
    wgt = np.zeros((image_h, image_w), dtype=np.float64)
    for x, y, mask in patch_masks:
        ph, pw = mask.shape
        weight = _distance_weight(ph, pw)
        acc[y : y + ph, x : x + pw] += mask.astype(np.float64) * weight
        wgt[y : y + ph, x : x + pw] += weight
    with np.errstate(divide="ignore", invalid="ignore"):
        stitched = np.where(wgt > 0, acc / wgt, 0)
    return stitched.astype(np.float32)


def run_patched_inference(
    model,
    image_path,
    prompt: str,
    patch_size: int = 1024,
    overlap: int = 256,
    batch_size: int = 1,
    show_patch_grid: bool = False,
    show_overlay: bool = False,
    save_path = None,
    mask_threshold: float = 0.5,
    alpha: float = 0.4,
    figsize: tuple = (18, 6),
):
    """Run patch-wise SAM3 inference on a large image and stitch the predicted masks.

    Args:
        model: Loaded SAM3 model.
        image_path: Path to the full image.
        prompt: Text prompt passed to the model.
        patch_size: Size of each square patch.
        overlap: Overlap between adjacent patches.
        batch_size: Number of patches processed per iteration. The current processor
            handles one image at a time, so this mainly controls progress chunking.
        show_patch_grid: If True, display the patch grid on the input image.
        show_overlay: If True, overlay the stitched mask on the original image.
        save_path: Optional path to save the result figure.
        mask_threshold: Threshold for the stitched soft mask.
        alpha: Opacity of the mask overlay.
        figsize: Figure size for matplotlib.

    Returns:
        Binary numpy mask of shape (H, W).
    """
    image = Image.open(image_path).convert("RGB")
    w, h = image.size

    patches = _extract_patches(image, patch_size, overlap)

    if show_patch_grid:
        fig, ax = plt.subplots(1, 1, figsize=(10, 10))
        ax.imshow(image)
        for i, (x, y, patch) in enumerate(patches):
            rect = mpatches.Rectangle(
                (x, y), patch.width, patch.height,
                linewidth=2, edgecolor="red", facecolor="none"
            )
            ax.add_patch(rect)
            ax.text(
                x + 5, y + 15, str(i),
                color="red", fontsize=10, backgroundcolor="white"
            )
        ax.set_title(f"Patch grid ({len(patches)} patches, size={patch_size}, overlap={overlap})")
        ax.axis("off")
        plt.tight_layout()
        if save_path is not None:
            save_path = Path(save_path)
            grid_path = save_path.parent / (save_path.stem + "_patch_grid" + save_path.suffix)
            plt.savefig(grid_path, bbox_inches="tight", dpi=150)
            print(f"Saved patch grid to {grid_path}")
        plt.show()

    processor = Sam3Processor(model, confidence_threshold=0.5)
    patch_masks = []

    for batch_start in range(0, len(patches), batch_size):
        batch = patches[batch_start : batch_start + batch_size]
        for x, y, patch in batch:
            inference_state = processor.set_image(patch)
            processor.reset_all_prompts(inference_state)
            inference_state = processor.set_text_prompt(
                state=inference_state, prompt=prompt
            )
            masks = inference_state.get("pred_masks", [])
            if masks:
                patch_mask = np.any(np.stack(masks, axis=0), axis=0).astype(np.float32)
            else:
                patch_mask = np.zeros((patch.height, patch.width), dtype=np.float32)
            patch_masks.append((x, y, patch_mask))

    stitched = _stitch_patch_masks(patch_masks, h, w)
    binary_mask = (stitched >= mask_threshold).astype(np.uint8)

    # Visualization
    image_np = np.array(image)
    ncols = 2 if show_overlay else 1
    fig, axes = plt.subplots(1, ncols, figsize=figsize)
    if ncols == 1:
        axes = [axes]

    axes[0].imshow(binary_mask, cmap="gray")
    axes[0].set_title("Stitched mask")
    axes[0].axis("off")

    if show_overlay:
        overlay = image_np.copy()
        color = np.array([255, 0, 0], dtype=np.uint8)
        mask_3ch = np.stack([binary_mask] * 3, axis=-1)
        overlay = np.where(
            mask_3ch > 0,
            (overlay * (1 - alpha) + color * alpha).astype(np.uint8),
            overlay
        )
        axes[1].imshow(overlay)
        axes[1].set_title("Overlay")
        axes[1].axis("off")

    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, bbox_inches="tight", dpi=150)
        print(f"Saved result to {save_path}")
    plt.show()

    return binary_mask
    


In [ ]:
# Example: run patched inference on a large image
result = run_patched_inference(
    model=model,
    image_path=imgs[idx],  # replace with any large image path
    prompt="thick_wire_panda",
    patch_size=512,
    overlap=256,
    batch_size=1,
    show_patch_grid=True,
    show_overlay=True,
    save_path="/tmp/patched_inference_result.jpg",
)

# Stratus 

In [ ]:
data_ = Path(r'/home/hpc_storage/data/projects/farken/stratus_new/data/Train/imgs_rescaled')
idx = np.random.randint(0, len(data_.ls()) - 1)
idx

In [ ]:
images = data_.ls(file_exts=['.png'])



In [ ]:

run_inference(model, images[idx], prompt="thin_wire_HD")

In [ ]:
# Permi

In [ ]:
images_ = Path(r'/home/hpc_storage/data/projects/farken/stratus_new/data/Train/imgs_rescaled/class1').ls()
sm_img = images_[0]

In [ ]:
pa_st = _extract_patches(Image.open(sm_img), patch_size=256, overlap=128)

In [ ]:
def plot_all_patches(patches, max_cols=4, figsize_per_patch=4):
    """Plot all patches in a grid with patch numbers.

    Args:
        patches: List of (x, y, patch) tuples as returned by _extract_patches.
        max_cols: Maximum number of columns in the grid.
        figsize_per_patch: Size of each patch subplot in inches.

    Returns:
        matplotlib.figure.Figure
    """
    n = len(patches)
    n_cols = min(max_cols, n)
    n_rows = (n + n_cols - 1) // n_cols

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(n_cols * figsize_per_patch, n_rows * figsize_per_patch),
    )
    if n == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if n_rows == 1 else axes.flatten()

    for ax in axes[n:]:
        ax.axis("off")

    for i, (x, y, patch) in enumerate(patches):
        axes[i].imshow(patch)
        axes[i].set_title(f"Patch {i}\n(x={x}, y={y})", fontsize=10)
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()
    return fig

In [ ]:
plot_all_patches(pa_st, max_cols=4, figsize_per_patch=3)

In [ ]:
run_inference(model, pa_st[106][2], prompt="thick wire HD")

In [ ]:
# Parmi

In [ ]:
p_path = Path(r'/home/hpc_storage/data/projects/userB/data/Permi_segmentation/')
sm_img = p_path.ls()[0]

In [ ]:
per_pa = _extract_patches(Image.open(sm_img), patch_size=256, overlap=128)

In [ ]:
plot_all_patches(per_pa, max_cols=10, figsize_per_patch=3)

In [ ]:
#run_inference(model, per_pa[105][2], prompt="thick wire panda")

In [ ]:
anno_path = Path(r'/home/hpc_storage/data/projects/userB/sam3_data/9cat_separated_wires_v3_unique_panda/train/annotation.json')

In [ ]:
coco = anno_path.read_text()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()